# T2.5 파일럿 베이스라인 1시드 학습 + 진단 4종 (Colab T4)

역할: **브랜치 가져오기 → 데이터 복사·해제 → `scripts/train_pilot_baseline.py` → `scripts/run_pilot_diagnosis.py` → `results/pilot/` 커밋·push**.
로직은 `src/sr_pipeline/`에 있고 이 노트북에는 넣지 않습니다.

- 이 학습은 **진단용 파일럿**입니다. 결과 표·집계에 넣지 않고 `results/pilot/`에만 둡니다.
- 에폭·에폭당 패치·배치·학습률은 `pilot_determined`가 `null`인 동안 코드에 기본값이 없습니다. 셀 1에서 정해 넣으세요(`None`이면 중단).
- 결과는 **본인 이름 브랜치**에 push합니다. main은 PR로만 바뀝니다(AGENTS.md).
- 체크포인트(`*.pth`)는 Drive에 두고 커밋하지 않습니다. 저장소가 Public이라 링크·토큰·경로를 출력이나 커밋에 남기지 않습니다. 실행 후 셀 출력은 지우고 커밋합니다.

In [ ]:
# 1. 실행 설정 (본인 값으로 채운다. None이면 중단한다)
REPO_URL = "https://github.com/minyi-k03/VideoProcessing-Project.git"
BRANCH = None              # 본인 이름 브랜치 (예: "YongMin"). main은 PR로만 바뀌므로 쓰지 않는다
SEED = 0                   # 파일럿은 1시드
EPOCHS = None
PATCHES_PER_EPOCH = None
BATCH_SIZE = None
LEARNING_RATE = None
EDGE_TOP_Q = None          # 에지 PSNR 상위 비율 q(%), 확정 전 파일럿 값
DATA_ZIP = None            # Drive의 압축 파일 경로 (예: /content/drive/MyDrive/<폴더>/<patch_pool_id>.zip)
DATA_DIR = "/content/data"
CKPT_PATH = None           # Drive 경로 (예: /content/drive/MyDrive/<폴더>/pilot_baseline_s0.pth), 커밋하지 않는다
SR_DIR = "/content/pilot_sr"  # 이미 있으면 스크립트가 거부한다(이전 SR 결과와 섞이지 않게)
missing = [k for k, v in dict(BRANCH=BRANCH, EPOCHS=EPOCHS, PATCHES_PER_EPOCH=PATCHES_PER_EPOCH, BATCH_SIZE=BATCH_SIZE,
           LEARNING_RATE=LEARNING_RATE, EDGE_TOP_Q=EDGE_TOP_Q, DATA_ZIP=DATA_ZIP, CKPT_PATH=CKPT_PATH).items() if v is None]
assert not missing, f"값을 채우세요: {missing}"
assert BRANCH != "main", "결과는 본인 이름 브랜치에 push한다. main은 PR로만 바뀐다(AGENTS.md)."


In [ ]:
# 2. 브랜치 가져오기
!git clone -b "{BRANCH}" "{REPO_URL}" /content/VideoProcessing-Project
%cd /content/VideoProcessing-Project
!pip install -q -r requirements.txt

In [ ]:
# 3. 데이터 준비: Drive 마운트 → 압축 파일을 런타임 로컬로 복사 → 해제 (SPEC 실행 창구, ADR-0005)
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "{DATA_DIR}"
!cp "{DATA_ZIP}" /content/pilot_data.zip
!unzip -q -o /content/pilot_data.zip -d "{DATA_DIR}"


In [ ]:
# 4. 학습 + SR 결과 생성 (T4). 학습 전에 테스트셋·SR 폴더를 먼저 확인한다.
# 학습 설정(코드에 고정한 옵티마이저·스케줄·증강 포함)은 results/pilot/pilot_train_settings.json에 기록된다.
!PYTHONPATH=src python scripts/train_pilot_baseline.py --data-dir "{DATA_DIR}" --checkpoint-path "{CKPT_PATH}" --sr-dir "{SR_DIR}" --output-dir results/pilot --seed {SEED} --epochs {EPOCHS} --patches-per-epoch {PATCHES_PER_EPOCH} --batch-size {BATCH_SIZE} --learning-rate {LEARNING_RATE}

In [ ]:
# 5. 진단 4종 (오차 지도, 에지·평탄 PSNR(Y), 주파수 스펙트럼, YCbCr 채널별 오차 + 최악 영상)
!PYTHONPATH=src python scripts/run_pilot_diagnosis.py --sr-dir "{SR_DIR}" --hr-dir "{DATA_DIR}/testsets" --output-dir results/pilot --edge-top-q {EDGE_TOP_Q}

In [ ]:
# 6. results/pilot/ 커밋·push (ADR-0005, 노트북 01 셀 3-1과 같은 방식)
# - 토큰은 getpass로만 받고 명령행·remote URL·파일·셀 출력에 남기지 않는다(GIT_ASKPASS 임시 스크립트로 전달).
# - 커밋 작성자 이름·이메일은 이 런타임의 명령에만 쓰고 출력하지 않는다.
# - git 명령이 하나라도 실패하면 멈춘다(커밋이 실패했는데 push가 성공처럼 보이는 것을 막는다).
# - 실행 후 이 노트북의 셀 출력은 지우고 커밋한다.
import getpass, os, stat, subprocess, tempfile

GH_USER = input("GitHub 사용자명: ").strip()
GIT_NAME = input("커밋 작성자 이름 (이 런타임에서만 사용): ").strip()
GIT_EMAIL = input("커밋 작성자 이메일 (이 런타임에서만 사용): ").strip()
token = getpass.getpass("GitHub 토큰 (저장되지 않음): ")

askpass = tempfile.NamedTemporaryFile("w", suffix=".sh", delete=False)
askpass.write('#!/bin/sh\ncase "$1" in\n  Username*) echo "$GH_USER";;\n  *) echo "$GH_TOKEN";;\nesac\n')
askpass.close()
os.chmod(askpass.name, stat.S_IRWXU)

env = {**os.environ, "GIT_ASKPASS": askpass.name, "GIT_TERMINAL_PROMPT": "0",
       "GH_USER": GH_USER, "GH_TOKEN": token}
ident = ["-c", f"user.name={GIT_NAME}", "-c", f"user.email={GIT_EMAIL}"]


def git(*args):
    shown = " ".join(a for a in args if not a.startswith("user."))  # 이름·이메일은 출력하지 않는다
    r = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    print(f"$ git {shown}  -> exit {r.returncode}")
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.stderr.strip():
        print(r.stderr.strip())
    if r.returncode != 0:
        raise RuntimeError(f"git {shown} 실패")
    return r


try:
    git("add", "results/pilot")
    git("status", "--short", "results/pilot")
    git(*ident, "commit", "-m", f"results: T2.5 pilot baseline seed {SEED} diagnosis")
    git("push", "origin", f"HEAD:{BRANCH}")
    print("✅ push 완료")
finally:
    os.remove(askpass.name)
    del token, env
